In [ ]:
import pandas as pd
import re
import html
from sklearn.model_selection import train_test_split



**Text Cleaning Function**

In [ ]:
def light_clean_for_bert(text):
    if pd.isna(text):
        return ""

    # 1. Unescape html anomalies (e.g., handles raw web artifacts like '&amp;' or '&gt;')
    text = html.unescape(str(text))

    # 2. Strip standard html brackets if present
    text = re.sub(r'<[^>]+>', ' ', text)

    # 3. Collapse multiple whitespace blocks and newlines into unified layout
    text = " ".join(text.split())

    return text





In [ ]:
# Loading the dataset
df = pd.read_csv('mental_health_combined_test.csv')
df.head()



,text,status
0,i don't understand whats wrong with me. i don'...,Anxiety
1,usually when i have anxiety just chatting with...,Anxiety
2,"well, i've had anxiety and panic syndrome for ...",Anxiety
3,"for the most minimal of things, like standing ...",Anxiety
4,i stay away from family and live with my roomm...,Anxiety


In [ ]:
# Appling the light text cleanup
df['text'] = df['text'].apply(light_clean_for_bert)

In [ ]:
# Translating the target categories into numeric IDs
label_dict = {'Anxiety': 0, 'Depression': 1, 'Normal': 2, 'Suicidal': 3}
df['label'] = df['status'].map(label_dict)


In [ ]:
# Dividing into training (80%) and validation/testing arrays (20%)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label'] # Guarantees perfectly uniform 4-class distributions across groups
)

In [ ]:
print(f"Dataset successfully loaded. Training items: {len(train_texts)} | Testing items: {len(test_texts)}")

Dataset successfully loaded. Training items: 793 | Testing items: 199


## **Step 3: Instantiate Tokenizer and the MentalBERT Model**

Instead of using generic word counters, we load MentalBERT, which understands clinical and mental health language habits natively out of the box.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Using standard, token-free public bert model path
model_name = "bert-base-uncased"

# Loading processing tokenizer parameters
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Downloading structural model parameters initialized for the 4 mental health categories
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# Shifting matrix execution operations onto the active GPU hardware environment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model properties successfully pulled and tracking over: {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model properties successfully pulled and tracking over: cuda


## **Step 4: Map Features to PyTorch Datasets**
Processes text chunks into indexing arrays for the learning loop.

In [ ]:
class MentalHealthDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # Tokenize structural details using the open-source dictionary matrix
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Flattening batch dimensions out of indexing operations
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)

        return item

# Packing processing layers together
train_dataset = MentalHealthDataset(train_texts, train_labels, tokenizer)
test_dataset = MentalHealthDataset(test_texts, test_labels, tokenizer)

print("Text sequences successfully packaged into Tensor dataset arrays.")

Text sequences successfully packaged into Tensor dataset arrays.


## **Step 5: Establishing Evaluation Metrics Tracker**
Handles tracking precision, recall, and F1 calculations.

In [ ]:
import evaluate
import numpy as np

# Loading the separate metric engines individually
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # 1. Computing accuracy (No 'average' parameter allowed here)
    acc_results = accuracy_metric.compute(predictions=predictions, references=labels)

    # 2. Computing the other metrics using 'macro' averaging for your 4 classes
    f1_results = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    precision_results = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    recall_results = recall_metric.compute(predictions=predictions, references=labels, average="macro")

    # 3. Combining everything into a single dictionary for the Trainer to read
    return {
        "accuracy": acc_results["accuracy"],
        "f1": f1_results["f1"],
        "precision": precision_results["precision"],
        "recall": recall_results["recall"]
    }

## **Step 6: Configuring and Run Training Loop**

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./results',          # Checkpoint log repository path
    num_train_epochs=3,              # Cycle iterations
    per_device_train_batch_size=16,  # Batch sample restrictions
    per_device_eval_batch_size=16,
    eval_strategy="epoch",           # CHANGED FROM evaluation_strategy TO eval_strategy
    save_strategy="epoch",
    learning_rate=2e-5,              # Fine-tuning step limitation rate
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,     # Select parameters from peak historical run state
    metric_for_best_model="f1"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# KICK OFF THE SYSTEM FINE-TUNING
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.114928,1.075123,0.547739,0.543139,0.561048,0.547347
2,0.800261,0.876660,0.658291,0.665017,0.690033,0.658265
3,0.624578,0.815325,0.678392,0.683692,0.694303,0.677959


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=150, training_loss=0.9338962682088217, metrics={'train_runtime': 211.0592, 'train_samples_per_second': 11.272, 'train_steps_per_second': 0.711, 'total_flos': 312976220424192.0, 'train_loss': 0.9338962682088217, 'epoch': 3.0})

## **Step 7: Zip, Mount and Save to Google Drive**

In [ ]:
import shutil
from google.colab import drive

output_dir = "./mental_bert_final_model"

# Saving the highest-performing weights and tokenizer vocabulary
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Packing everything into a unified zip archive file
shutil.make_archive('mental_bert_final_model', 'zip', output_dir)

# Mounting Google Drive and safely copy the file over
drive.mount('/content/drive')
!cp mental_bert_final_model.zip /content/drive/MyDrive/

print("Complete! trained BERT model zip package is safely resting in the Google Drive.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Complete! trained BERT model zip package is safely resting in the Google Drive.


## **Step 8: Live Real-Time Testing Function**

In [ ]:
import torch
import torch.nn.functional as F
import html
import re

# 1. Bringing in the exact text normalization rule from the pipeline
def predict_mental_health_status(input_text, model, tokenizer, device):
    # Applying the light cleaning rule
    cleaned_text = html.unescape(str(input_text))
    cleaned_text = re.sub(r'<[^>]+>', ' ', cleaned_text)
    cleaned_text = " ".join(cleaned_text.split())

    # 2. Tokenizing the input sentence
    inputs = tokenizer(
        cleaned_text,
        truncation=True,
        padding='max_length',
        max_length=256,
        return_tensors="pt"
    )

    # Moving tensor keys to the GPU
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # 3. Passing through the model (disable gradient calculations to save memory)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        # Calculating raw probability percentages (0% to 100%) for each class
        probabilities = F.softmax(logits, dim=-1).squeeze().cpu().numpy()

    # 4. Mapping the predicted ID back to the exact class text
    class_labels = ['Anxiety', 'Depression', 'Normal', 'Suicidal']
    predicted_class_id = torch.argmax(logits, dim=-1).item()
    predicted_label = class_labels[predicted_class_id]

    # Printing the clean diagnostic summary report
    print("=" * 50)
    print(f"INPUT TEXT: '{input_text}'")
    print("-" * 50)
    print(f"--> PREDICTED STATUS: **{predicted_label}**")
    print("-" * 50)
    print("CONFIDENCE DISTRIBUTION BREAKDOWN:")
    for label, prob in zip(class_labels, probabilities):
        print(f"   • {label.ljust(12)}: {prob * 100:.2f}%")
    print("=" * 50)

    return predicted_label


# LIVE TEST BENCH BENCHMARK


# Example text to evaluate
sample_sentence = "I've been feeling deeply overwhelmed lately and my chest feels tight every single night."

# Running the function using the freshly trained model assets
prediction = predict_mental_health_status(sample_sentence, model, tokenizer, device)

INPUT TEXT: 'I've been feeling deeply overwhelmed lately and my chest feels tight every single night.'
--------------------------------------------------
--> PREDICTED STATUS: **Anxiety**
--------------------------------------------------
CONFIDENCE DISTRIBUTION BREAKDOWN:
   • Anxiety     : 51.12%
   • Depression  : 30.46%
   • Normal      : 10.06%
   • Suicidal    : 8.36%
